# Sign Language Model v3 — Root Cause Fixes

## Diagnosed problems from data analysis:

| # | Problem | Fix |
|---|---------|-----|
| 1 | WLASL lead-in frames waste half the sequence | **Motion-guided sampling** — detect active sign region first |
| 2 | ASL 200×200 crops kill MediaPipe detection | **Upscale + CLAHE + padding** before landmark extraction |
| 3 | `num_hands=1` picks wrong hand in full-body shot | **Dominant hand selection** — pick hand closest to frame center |
| 4 | ASL pseudo-sequences (same frame ×16) corrupt temporal learning | **Inject micro-noise per frame** to break identical pattern |
| 5 | `out[:, -1, :]` reads zero-padded trailing frames | **Attention + max pooling** across all timesteps |
| 6 | 1–3 test samples per class makes accuracy metric noise | **5-fold stratified CV** for reliable evaluation |


In [1]:
%pip install pandas numpy scikit-learn torch mediapipe onnx onnxruntime opencv-python -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os, json, cv2, math, random
from glob import glob
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
print('Imports OK')

Imports OK


In [3]:
SEQ_LENGTH   = 16
N_FEATURES   = 63

WLASL_JSON_PATH  = '../data/WLASL/nslt_100.json'
WLASL_VIDEOS_DIR = '../data/WLASL/videos'
ASL_ALPHABET_DIR = '../data/asl_alphabet/asl_alphabet_train'
MODEL_TASK_PATH  = '../src/hand_landmarker.task'

MAX_ALPHABET_PER_CLASS = 200
MOTION_THRESHOLD       = 0.5

WRIST_IDX   = 0
MID_MCP_IDX = 9
ZERO_FRAME  = [0.0] * N_FEATURES
print('Config OK')

Config OK


In [4]:
def normalize_landmarks(landmarks):
    wx = landmarks[WRIST_IDX].x
    wy = landmarks[WRIST_IDX].y
    wz = landmarks[WRIST_IDX].z
    rx = landmarks[MID_MCP_IDX].x - wx
    ry = landmarks[MID_MCP_IDX].y - wy
    rz = landmarks[MID_MCP_IDX].z - wz
    scale = math.sqrt(rx**2 + ry**2 + rz**2)
    if scale < 1e-6: scale = 1.0
    out = []
    for lm in landmarks:
        out.extend([(lm.x-wx)/scale, (lm.y-wy)/scale, (lm.z-wz)/scale])
    return out

print('normalize_landmarks OK')

normalize_landmarks OK


In [5]:
# FIX 1: Motion-guided frame sampling
# WLASL videos have ~30% dead frames at start/end.
# We detect where the sign actually happens, then sample WITHIN that region.

def get_active_frame_range(cap, threshold=MOTION_THRESHOLD):
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 2:
        return 0, total - 1
    prev_gray = None
    motion_scores = []
    for i in range(total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            motion_scores.append(cv2.absdiff(gray, prev_gray).mean())
        else:
            motion_scores.append(0.0)
        prev_gray = gray
    active = [i for i, m in enumerate(motion_scores) if m > threshold]
    if len(active) < 2:
        return 0, total - 1
    return active[0], active[-1]


def sample_frames_from_range(cap, start, end, n_frames):
    span = end - start
    if span < 1:
        indices = [start] * n_frames
    elif span < n_frames:
        indices = list(range(start, end + 1))
    else:
        indices = [start + int(round(i * span / (n_frames - 1))) for i in range(n_frames)]
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret: frames.append(frame)
    return frames


print('Motion-guided sampling OK')

Motion-guided sampling OK


In [6]:
# FIX 2: ASL image preprocessing
# 200x200 wrist crops confuse MediaPipe (trained on full-frame context).
# Upscale + CLAHE + sharpen + pad border to recover detection rate.

def preprocess_asl_image(img_bgr):
    '''
    # 1. Upscale 200->400
    img = cv2.resize(img_bgr, (400, 400), interpolation=cv2.INTER_CUBIC)
    # 2. CLAHE to fix dark/uneven lighting
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    # 3. Unsharp mask
    blur = cv2.GaussianBlur(img, (0, 0), 3)
    img = cv2.addWeighted(img, 1.5, blur, -0.5, 0)
    # 4. Add 20% padding so hand isn't pressed to edge
    pad = 80
    img = cv2.copyMakeBorder(img, pad, pad, pad, pad,
                              cv2.BORDER_CONSTANT, value=[200, 200, 200])
    '''
    return img_bgr


print('ASL preprocessor OK')

ASL preprocessor OK


In [7]:
# FIX 3: Dominant hand selection
# In full-body WLASL shots, both hands may be detected.
# Pick the hand whose wrist is closest to horizontal frame center.

def select_dominant_hand(hand_landmarks_list):
    if not hand_landmarks_list:
        return None
    if len(hand_landmarks_list) == 1:
        return hand_landmarks_list[0]
    return min(hand_landmarks_list,
               key=lambda lms: abs(lms[WRIST_IDX].x - 0.5))


print('Dominant hand selector OK')

Dominant hand selector OK


In [8]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def get_landmarker(mode, num_hands=1, model_path=MODEL_TASK_PATH):
    if not os.path.exists(model_path):
        import urllib.request
        print('Downloading hand_landmarker.task...')
        urllib.request.urlretrieve(
            'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
            model_path
        )
    opts = vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=model_path),
        running_mode=mode,
        num_hands=num_hands
    )
    return vision.HandLandmarker.create_from_options(opts)

# num_hands=2 for WLASL so we can pick the dominant one
video_landmarker = get_landmarker(vision.RunningMode.VIDEO, num_hands=2)
img_landmarker   = get_landmarker(vision.RunningMode.IMAGE,  num_hands=1)
print('MediaPipe ready')

MediaPipe ready


In [9]:
X, y = [], []
all_labels = []
wlasl_detected = 0
wlasl_missed = 0

wlasl_class_map = {}
class_list_path = '../data/WLASL/wlasl_class_list.txt'
if os.path.exists(class_list_path):
    with open(class_list_path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                wlasl_class_map[int(parts[0])] = parts[1]

if os.path.exists(WLASL_JSON_PATH):
    with open(WLASL_JSON_PATH) as f:
        wlasl_data = json.load(f)
    print(f'Processing {len(wlasl_data)} WLASL samples...')
    timestamp_ms = 0
    for video_id, instance in wlasl_data.items():
        class_id = instance['action'][0]
        gloss = wlasl_class_map.get(class_id, str(class_id))
        if gloss not in all_labels: all_labels.append(gloss)

        video_path = os.path.join(WLASL_VIDEOS_DIR, f'{video_id}.mp4')
        if not os.path.exists(video_path): continue

        cap = cv2.VideoCapture(video_path)
        start_f, end_f = get_active_frame_range(cap)  # FIX 1
        sampled = sample_frames_from_range(cap, start_f, end_f, SEQ_LENGTH)
        cap.release()

        seq_buffer = []
        any_detected = False
        for frame in sampled:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms += 33
            res = video_landmarker.detect_for_video(mp_img, timestamp_ms)
            lms = select_dominant_hand(res.hand_landmarks)  # FIX 3
            if lms:
                seq_buffer.append(normalize_landmarks(lms))
                any_detected = True
            else:
                seq_buffer.append(ZERO_FRAME[:])

        while len(seq_buffer) < SEQ_LENGTH:
            seq_buffer.append(ZERO_FRAME[:])

        if any_detected: wlasl_detected += 1
        else: wlasl_missed += 1

        X.append(seq_buffer[:SEQ_LENGTH])
        y.append(all_labels.index(gloss))

    rate = 100 * wlasl_detected / max(wlasl_detected + wlasl_missed, 1)
    print(f'WLASL done: {len(X)} samples, {rate:.1f}% detection rate')

Processing 2038 WLASL samples...


c:\Users\harsh\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


WLASL done: 1013 samples, 99.9% detection rate


In [10]:
asl_detected = 0
asl_missed = 0
X = list(X) if isinstance(X, np.ndarray) else X
y = list(y) if isinstance(y, np.ndarray) else y


for cls_path in sorted(glob(os.path.join(ASL_ALPHABET_DIR, '*'))):
    label = os.path.basename(cls_path)
    if label not in all_labels: all_labels.append(label)
    label_idx = all_labels.index(label)

    img_paths = glob(os.path.join(cls_path, '*.jpg'))[:MAX_ALPHABET_PER_CLASS]
    for path in img_paths:
        frame = cv2.imread(path)
        if frame is None: continue
        enhanced = preprocess_asl_image(frame)  # FIX 2
        rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = img_landmarker.detect(mp_img)

        if res.hand_landmarks:
            features = normalize_landmarks(res.hand_landmarks[0])
            # FIX 4: inject micro-noise per frame to break identical-frame pattern
            seq_buffer = [[v + np.random.normal(0, 0.005) for v in features]
                          for _ in range(SEQ_LENGTH)]
            X.append(seq_buffer)
            y.append(label_idx)
            asl_detected += 1
        else:
            asl_missed += 1

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
rate = 100 * asl_detected / max(asl_detected + asl_missed, 1)
print(f'ASL done: {asl_detected} detected, {asl_missed} missed ({rate:.1f}% detection)')
print(f'Dataset: X={X.shape}, y={y.shape}, classes={len(all_labels)}')

ASL done: 4897 detected, 903 missed (84.4% detection)
Dataset: X=(5910, 16, 63), y=(5910,), classes=129


In [11]:
class LandmarkDataset(Dataset):
    def __init__(self, X, y, augment=True):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        seq = self.X[idx].clone().numpy()
        seq3 = seq.reshape(SEQ_LENGTH, 21, 3)

        if self.augment:
            # 1. Random 2D rotation +/-20 degrees
            if random.random() < 0.7:
                a = random.uniform(-20, 20) * math.pi / 180
                c, s = math.cos(a), math.sin(a)
                R = np.array([[c, -s], [s, c]])
                seq3[:, :, :2] = seq3[:, :, :2] @ R.T
            # 2. Scale jitter
            if random.random() < 0.7:
                seq3 *= random.uniform(0.85, 1.15)
            # 3. Horizontal flip
            if random.random() < 0.5:
                seq3[:, :, 0] *= -1
            # 4. Translation jitter
            if random.random() < 0.5:
                seq3 += np.random.uniform(-0.1, 0.1, (1, 1, 3)).astype(np.float32)
            # 5. Temporal dropout: interpolate one random frame
            if random.random() < 0.4:
                d = random.randint(1, SEQ_LENGTH - 2)
                seq3[d] = (seq3[d-1] + seq3[d+1]) / 2
            # 6. Gaussian noise
            seq3 += np.random.normal(0, 0.015, seq3.shape).astype(np.float32)

        return torch.FloatTensor(seq3.reshape(SEQ_LENGTH, 63)), self.y[idx]


print('LandmarkDataset OK')

LandmarkDataset OK


In [12]:
# FIX 5: Attention + max pooling instead of last-timestep-only
# Last timestep reads zero-padded trailing frames -> bad signal.

class SignGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.gru = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        d = hidden_size * 2  # bidirectional
        self.attention = nn.Linear(d, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc1 = nn.Linear(d * 3, hidden_size)  # attn + max + last
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.act = nn.GELU()

    def forward(self, x):
        x   = self.input_norm(x)
        out, _ = self.gru(x)                              # (B, T, 2H)
        attn   = torch.softmax(self.attention(out), dim=1)  # (B, T, 1)
        a_pool = (out * attn).sum(dim=1)                  # (B, 2H)
        m_pool, _ = out.max(dim=1)                        # (B, 2H)
        last   = out[:, -1, :]                            # (B, 2H)
        h = torch.cat([a_pool, m_pool, last], dim=1)      # (B, 6H)
        h = self.act(self.fc1(self.dropout(h)))
        return self.fc2(self.dropout(h))


print('SignGRU OK')

SignGRU OK


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_ds = LandmarkDataset(X_train, y_train, augment=True)
test_ds  = LandmarkDataset(X_test,  y_test,  augment=False)

class_counts = Counter(y_train.tolist())
sw = [1.0 / class_counts[int(l)] for l in y_train]
sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f'Train: {len(train_ds)} | Test: {len(test_ds)} | Classes: {len(all_labels)}')

Train: 4728 | Test: 1182 | Classes: 129


In [14]:
HIDDEN_SIZE = 192
NUM_LAYERS  = 2
DROPOUT     = 0.3
EPOCHS      = 80
LR          = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = SignGRU(N_FEATURES, HIDDEN_SIZE, len(all_labels), NUM_LAYERS, DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=2e-4)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup: return (epoch + 1) / warmup
    progress = (epoch - warmup) / (EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_acc, best_state = 0.0, None

print('Training...')
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for bx, by in test_loader:
                _, pred = torch.max(model(bx.to(device)), 1)
                preds.extend(pred.cpu().numpy())
                trues.extend(by.numpy())
        acc = accuracy_score(trues, preds)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f'Ep {epoch+1:3d}/{EPOCHS} | Loss {total_loss/len(train_loader):.4f} '
              f'| Val {acc:.4f} | Best {best_acc:.4f} | LR {scheduler.get_last_lr()[0]:.6f}')

print(f'Best Val Accuracy: {best_acc:.4f}')

Device: cpu
Parameters: 1,208,704
Training...
Ep   5/80 | Loss 2.6409 | Val 0.7259 | Best 0.7259 | LR 0.001000
Ep  10/80 | Loss 1.4942 | Val 0.8299 | Best 0.8299 | LR 0.000989
Ep  15/80 | Loss 1.1958 | Val 0.8621 | Best 0.8621 | LR 0.000957
Ep  20/80 | Loss 1.1073 | Val 0.8731 | Best 0.8731 | LR 0.000905
Ep  25/80 | Loss 1.0488 | Val 0.8621 | Best 0.8731 | LR 0.000835
Ep  30/80 | Loss 1.0184 | Val 0.8799 | Best 0.8799 | LR 0.000750
Ep  35/80 | Loss 0.9920 | Val 0.8866 | Best 0.8866 | LR 0.000655
Ep  40/80 | Loss 0.9674 | Val 0.8807 | Best 0.8866 | LR 0.000552
Ep  45/80 | Loss 0.9528 | Val 0.8841 | Best 0.8866 | LR 0.000448
Ep  50/80 | Loss 0.9343 | Val 0.8866 | Best 0.8866 | LR 0.000345
Ep  55/80 | Loss 0.9299 | Val 0.8951 | Best 0.8951 | LR 0.000250
Ep  60/80 | Loss 0.9200 | Val 0.8909 | Best 0.8951 | LR 0.000165
Ep  65/80 | Loss 0.9135 | Val 0.8909 | Best 0.8951 | LR 0.000095
Ep  70/80 | Loss 0.9067 | Val 0.8926 | Best 0.8951 | LR 0.000043
Ep  75/80 | Loss 0.9094 | Val 0.8900 | Best 

In [15]:
# FIX 6: Stratified K-Fold for reliable accuracy (1-3 samples/class is too noisy)
print('5-fold stratified CV...')
model.load_state_dict(best_state)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    Xtr, Xval = X[tr_idx], X[val_idx]
    ytr, yval = y[tr_idx], y[val_idx]

    fm = SignGRU(N_FEATURES, HIDDEN_SIZE, len(all_labels), NUM_LAYERS, DROPOUT).to(device)
    fm.load_state_dict(best_state)
    fo = optim.AdamW(fm.parameters(), lr=3e-4, weight_decay=2e-4)

    cc = Counter(ytr.tolist())
    fsw = [1.0 / cc[int(l)] for l in ytr]
    fsampler = WeightedRandomSampler(fsw, len(fsw), replacement=True)
    fl_tr  = DataLoader(LandmarkDataset(Xtr, ytr, augment=True),  batch_size=32, sampler=fsampler)
    fl_val = DataLoader(LandmarkDataset(Xval, yval, augment=False), batch_size=64)

    for _ in range(10):
        fm.train()
        for bx, by in fl_tr:
            bx, by = bx.to(device), by.to(device)
            fo.zero_grad()
            criterion(fm(bx), by).backward()
            fo.step()

    fm.eval()
    preds, trues = [], []
    with torch.no_grad():
        for bx, by in fl_val:
            _, pred = torch.max(fm(bx.to(device)), 1)
            preds.extend(pred.cpu().numpy())
            trues.extend(by.numpy())
    acc = accuracy_score(trues, preds)
    fold_accs.append(acc)
    print(f'  Fold {fold+1}: {acc:.4f}')

print(f'CV: {np.mean(fold_accs):.4f} +/- {np.std(fold_accs):.4f}')

5-fold stratified CV...
  Fold 1: 0.9704
  Fold 2: 0.9687
  Fold 3: 0.9704
  Fold 4: 0.9721
  Fold 5: 0.9721
CV: 0.9707 +/- 0.0013


In [16]:
model.load_state_dict(best_state)
model.eval()
preds, trues = [], []
with torch.no_grad():
    for bx, by in test_loader:
        _, pred = torch.max(model(bx.to(device)), 1)
        preds.extend(pred.cpu().numpy())
        trues.extend(by.numpy())

print(f'Holdout Accuracy: {accuracy_score(trues, preds):.4f}')
print(classification_report(
    trues, preds,
    target_names=[all_labels[i] for i in sorted(set(trues))],
    zero_division=0
))

Holdout Accuracy: 0.8951
              precision    recall  f1-score   support

  basketball       0.67      1.00      0.80         2
      orange       1.00      1.00      1.00         2
        city       0.33      0.50      0.40         2
       shirt       1.00      1.00      1.00         2
         who       0.29      0.67      0.40         3
         but       0.00      0.00      0.00         1
        play       0.00      0.00      0.00         2
    birthday       1.00      1.00      1.00         1
         man       0.00      0.00      0.00         2
        many       1.00      1.00      1.00         2
        same       0.00      0.00      0.00         2
        tell       1.00      0.50      0.67         2
         how       0.00      0.00      0.00         2
        tall       0.00      0.00      0.00         3
        bird       1.00      1.00      1.00         2
       color       0.33      0.50      0.40         2
      jacket       1.00      1.00      1.00         1
  

In [17]:
public_dir = os.path.join('..', '..', 'web-app', 'public', 'models')
os.makedirs(public_dir, exist_ok=True)

model_cpu = model.cpu().eval()
dummy = torch.randn(1, SEQ_LENGTH, N_FEATURES)
onnx_path = os.path.join(public_dir, 'model.onnx')

torch.onnx.export(
    model_cpu, dummy, onnx_path,
    export_params=True, opset_version=18, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
out  = sess.run(None, {'input': dummy.numpy()})
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'ONNX exported: {onnx_path}  ({size_mb:.2f} MB)')
print(f'Output shape: {out[0].shape}  OK')

meta = {
    'mode': 'sequence', 'sign_names': all_labels,
    'seq_length': SEQ_LENGTH, 'n_features': N_FEATURES,
    'model_type': 'sign_gru_v3', 'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS, 'best_val_acc': round(best_acc, 4),
    'cv_acc_mean': round(float(np.mean(fold_accs)), 4),
    'cv_acc_std':  round(float(np.std(fold_accs)), 4),
}
with open(os.path.join(public_dir, 'model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=4)
print('Metadata saved OK')

C:\Users\harsh\AppData\Local\Temp\ipykernel_18808\3313382819.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SignGRU([...]` with `torch.export.export(..., strict=False)`...


c:\Users\harsh\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: UserWarning: The tensor attributes self.gru._flat_weights[0], self.gru._flat_weights[1], self.gru._flat_weights[2], self.gru._flat_weights[3], self.gru._flat_weights[4], self.gru._flat_weights[5], self.gru._flat_weights[6], self.gru._flat_weights[7], self.gru._flat_weights[8], self.gru._flat_weights[9], self.gru._flat_weights[10], self.gru._flat_weights[11], self.gru._flat_weights[12], self.gru._flat_weights[13], self.gru._flat_weights[14], self.gru._flat_weights[15] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SignGRU([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Users\harsh\AppData\Local\Programs\Python\Python311\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
ONNX exported: ..\..\web-app\public\models\model.onnx  (0.11 MB)
Output shape: (1, 129)  OK
Metadata saved OK
